### Data Download FMP ###

In [1]:
# Read in the txt tickers file
txt_path = f'../../data/raw_data/tickers_448_v2.txt'
with open(txt_path, 'r') as f:
    tickers = f.read().splitlines()
    

In [2]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import logging

api_key = "6JSO2ou6k0WLT5KmhOBxLpkkKdX4idz1"

#!/usr/bin/env python
try:
    # For Python 3.0 and later
    from urllib.request import urlopen
except ImportError:
    # Fall back to Python 2's urllib2
    from urllib2 import urlopen

import certifi
import json

def get_jsonparsed_data(url):
    response = urlopen(url, cafile=certifi.where())
    data = response.read().decode("utf-8")
    return json.loads(data)



for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)





In [3]:
### additional functions ###
def fetch_all_insider_trades(ticker, api_key, transaction_type=None, max_pages=10):
    """
    Fetch all insider trades with pagination
    
    Args:
        ticker: Stock symbol
        api_key: FMP API key
        transaction_type: Optional filter (e.g., 'S-Sale', 'P-Purchase')
        max_pages: Maximum number of pages to fetch
    
    Returns:
        List of all insider trades
    """
    all_trades = []
    page = 0
    
    while page < max_pages:
        url = (
            f"https://financialmodelingprep.com/stable/insider-trading/search"
            f"?symbol={ticker}&page={page}&limit=100&apikey={api_key}"
        )
        
        if transaction_type:
            url += f"&transactionType={transaction_type}"
        
        try:
            data = get_jsonparsed_data(url)
            
            if not data:  # No more data
                break
            
            all_trades.extend(data)
            #print(f"Fetched page {page}: {len(data)} trades")
            
            if len(data) < 100:  # Last page
                break
            
            page += 1
        except Exception as e: 
            logging.error(f"Error fetching data for {ticker} on page {page}: {e}")
            break
    
    return all_trades

# # Usage
# for ticker in tickers:
#     print(f"\nFetching all insider trades for {ticker}...")
#     all_trades = fetch_all_insider_trades(ticker, api_key, max_pages=5)
    
#     if all_trades:
#         df = pd.DataFrame(all_trades)
#         print(f"Total trades fetched: {len(df)}")

In [7]:
from_date = "2000-01-01"
to_date = "2025-06-01"
period_length = 10
timeframe = "1day"

# full_income_statement = pd.DataFrame()
# full_balance_sheet = pd.DataFrame()
# full_cash_flow = pd.DataFrame()
# full_financial_ratios = pd.DataFrame()
#full_williams = pd.DataFrame()
#full_historical_analyst_grades = pd.DataFrame()
full_historical_ratings = pd.DataFrame()
# all_insider_trades = pd.DataFrame()

# Optionally read from existing parquets
# full_income_statement = pd.read_parquet('../../data/raw_data/full_income_statement.parquet')
# full_balance_sheet = pd.read_parquet('../../data/raw_data/full_balance_sheet.parquet')
# full_cash_flow = pd.read_parquet('../../data/raw_data/full_cash_flow.parquet')
# full_financial_ratios = pd.read_parquet('../../data/raw_data/full_financial_ratios.parquet')
# full_williams = pd.read_parquet('../../data/raw_data/full_williams.parquet')
# all_insider_trades = pd.read_parquet('../../data/raw_data/all_insider_trades.parquet')
#current_tickers_new = set(full_income_statement['symbol'].unique().tolist())
current_tickers_new = set([])

for ticker in tickers:
    if ticker in current_tickers_new: 
        continue
    
    logging.info(f"Processing ticker: {ticker}")
    
    # Financial Statements
    # income_statement_url = (f"https://financialmodelingprep.com/stable/income-statement?symbol={ticker}&apikey={api_key}&period=quarter&limit=100")
    # balance_sheet_url = (f"https://financialmodelingprep.com/stable/balance-sheet-statement?symbol={ticker}&apikey={api_key}&period=quarter&limit=100")
    # cash_flow_url = (f"https://financialmodelingprep.com/stable/cash-flow-statement?symbol={ticker}&apikey={api_key}&period=quarter&limit=100")
    
    # income_statement_data = get_jsonparsed_data(income_statement_url)
    # balance_sheet_data = get_jsonparsed_data(balance_sheet_url)
    # cash_flow_data = get_jsonparsed_data(cash_flow_url)
    
    # Financial Ratios
    # financial_ratios_url = f"https://financialmodelingprep.com/stable/ratios?symbol={ticker}&apikey={api_key}&period=annual&limit=100"
    # financial_ratios_data = get_jsonparsed_data(financial_ratios_url)
    #financial_ratios_data = None
    
    # Williams URL
    # williams_url_with_dates = (
    #     f"https://financialmodelingprep.com/stable/technical-indicators/williams"
    #     f"?symbol={ticker}&periodLength={period_length}&timeframe={timeframe}"
    #     f"&from={from_date}&to={to_date}&apikey={api_key}"
    # )
    # williams_data = pd.DataFrame(get_jsonparsed_data(williams_url_with_dates))
    # williams_data['symbol'] = ticker
    
    # Historical Analyst Grades
    historical_analyst_grades_url = (
        f"https://financialmodelingprep.com/stable/grades-historical?symbol={ticker}&apikey={api_key}&limit=10000000"
    )
    historical_analyst_grades_data = pd.DataFrame(get_jsonparsed_data(historical_analyst_grades_url))
    
    
    # Historical Ratings
    historical_ratings_url = (
        f'https://financialmodelingprep.com/stable/ratings-historical?symbol={ticker}&apikey={api_key}&limit=1000000'
    )
    historical_ratings_data = pd.DataFrame(get_jsonparsed_data(historical_ratings_url))
    
    # Insider Trades
    #all_trades = fetch_all_insider_trades(ticker, api_key, max_pages=1000)
    
    for data in [#income_statement_data, balance_sheet_data, cash_flow_data, financial_ratios_data, all_trades, 
                 #historical_analyst_grades_data, 
                 historical_ratings_data]:
                 #williams_data]:
        if data is not None and len(data) > 0:
            df = pd.DataFrame(data) if not isinstance(data, pd.DataFrame) else data
            logging.info(f"{ticker}: Retrieved {len(df)} records.")
            
            # Add to full DataFrames
            # if data == income_statement_data:
            #     full_income_statement = pd.concat([full_income_statement, df], ignore_index=True)
            # elif data == balance_sheet_data:
            #     full_balance_sheet = pd.concat([full_balance_sheet, df], ignore_index=True)
            # elif data == cash_flow_data:
            #     full_cash_flow = pd.concat([full_cash_flow, df], ignore_index=True)
            # elif data == financial_ratios_data:
            #     full_financial_ratios = pd.concat([full_financial_ratios, df], ignore_index=True)
            # if data is williams_data:
            #     full_williams = pd.concat([full_williams, df], ignore_index=True)
            # if data is historical_analyst_grades_data:
            #     full_historical_analyst_grades = pd.concat([full_historical_analyst_grades, df], ignore_index=True)
            # elif data == all_trades:
            #     all_insider_trades = pd.concat([all_insider_trades, df], ignore_index=True)
            if data is historical_ratings_data:
                full_historical_ratings = pd.concat([full_historical_ratings, df], ignore_index=True)
        else:
            logging.info(f"{ticker}: No data retrieved.")
            
    current_tickers_new.add(ticker)
    
    # print(get_jsonparsed_data(income_statement_url))
    # print(len(get_jsonparsed_data(income_statement_url)))
    
    
# Fetching economic data



2026-01-23 14:26:24 [INFO] Processing ticker: JNJ
C:\Users\yinki\AppData\Local\Temp\ipykernel_18192\2874180479.py:21: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())
2026-01-23 14:26:30 [INFO] JNJ: Retrieved 4507 records.
2026-01-23 14:26:30 [INFO] Processing ticker: AJG
2026-01-23 14:26:36 [INFO] AJG: Retrieved 5573 records.
2026-01-23 14:26:36 [INFO] Processing ticker: FI
2026-01-23 14:26:42 [INFO] FI: Retrieved 2610 records.
2026-01-23 14:26:42 [INFO] Processing ticker: TAP
2026-01-23 14:26:48 [INFO] TAP: Retrieved 5729 records.
2026-01-23 14:26:48 [INFO] Processing ticker: INTU
2026-01-23 14:26:54 [INFO] INTU: Retrieved 5958 records.
2026-01-23 14:26:54 [INFO] Processing ticker: TMUS
2026-01-23 14:27:00 [INFO] TMUS: Retrieved 4675 records.
2026-01-23 14:27:00 [INFO] Processing ticker: FIS
2026-01-23 14:27:06 [INFO] FIS: Retrieved 5890 records.
2026-01-23 14:27:06 [INFO] Processing ticke

In [ ]:
# Multiple indicators in one request (comma-separated)

full_economic_data = pd.DataFrame()

economic_indicators = [
    "GDP",
     "realGDP",
    "nominalPotentialGDP",
    "realGDPPerCapita",
    "federalFunds",
    "CPI",
    "inflationRate",
    "inflation",
    "retailSales",
    "consumerSentiment",
    "durableGoods",
    "unemploymentRate",
    "totalNonfarmPayroll",
    "initialClaims",
    "industrialProductionTotalIndex",
    "newPrivatelyOwnedHousingUnitsStartedTotalUnits",
    "totalVehicleSales",
    "retailMoneyFunds",
    "smoothedUSRecessionProbabilities",
    "3MonthOr90DayRatesAndYieldsCertificatesOfDeposit",
    "commercialBankInterestRateOnCreditCardPlansAllAccounts",
    "30YearFixedRateMortgageAverage",
    "15YearFixedRateMortgageAverage",
    "tradeBalanceGoodsAndServices"
]

for indicator in economic_indicators:
    logging.info(f"Fetching economic indicator: {indicator}")
    economic_indicator_url = (
        f"https://financialmodelingprep.com/stable/economic-indicators"
        f"?name={indicator}&from={from_date}&to={to_date}&apikey={api_key}"
    )
    economic_data = get_jsonparsed_data(economic_indicator_url)
    economic_data = pd.DataFrame(economic_data)
    economic_data.rename(columns={"value": indicator}, inplace=True)
    economic_data.drop(columns=["name"], inplace=True)
    
    if len(full_economic_data) == 0:
        full_economic_data = economic_data
    else:
        # Merge based on date
        full_economic_data = pd.merge(full_economic_data, economic_data, on="date", how="outer")
    
    
full_economic_data.sort_values(by="date").to_parquet('../../data/raw_data/full_economic_data.parquet')

2026-01-22 19:36:43 [INFO] Fetching economic indicator: GDP


C:\Users\yinki\AppData\Local\Temp\ipykernel_13752\2874180479.py:21: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())
2026-01-22 19:36:45 [INFO] Fetching economic indicator: realGDP
C:\Users\yinki\AppData\Local\Temp\ipykernel_13752\2874180479.py:21: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())
2026-01-22 19:36:47 [INFO] Fetching economic indicator: nominalPotentialGDP
C:\Users\yinki\AppData\Local\Temp\ipykernel_13752\2874180479.py:21: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url, cafile=certifi.where())
2026-01-22 19:36:48 [INFO] Fetching economic indicator: realGDPPerCapita
C:\Users\yinki\AppData\Local\Temp\ipykernel_13752\2874180479.py:21: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom co

AttributeError: 'NoneType' object has no attribute 'to_parquet'

In [27]:
full_economic_data.sort_values(by="date").to_parquet('../../data/raw_data/full_economic_data.parquet')

In [ ]:
# Save all to parquet
full_income_statement.to_parquet('../../data/raw_data/full_income_statement.parquet', index=False)
full_balance_sheet.to_parquet('../../data/raw_data/full_balance_sheet.parquet', index=False)
full_cash_flow.to_parquet('../../data/raw_data/full_cash_flow.parquet', index=False)
full_financial_ratios.to_parquet('../../data/raw_data/full_financial_ratios.parquet', index=False)
full_williams.to_parquet('../../data/raw_data/full_williams.parquet', index=False)
all_insider_trades.to_parquet('../../data/raw_data/all_insider_trades.parquet', index=False)  
full_historical_analyst_grades.to_parquet('../../data/raw_data/full_historical_analyst_grades.parquet', index=False)
full_historical_ratings.to_parquet('../../data/raw_data/full_historical_ratings.parquet', index=False)
#economic_data.to_parquet('../../data/raw_data/economic_data.parquet', index=False)

In [12]:
all_insider_trades.to_parquet('../../data/raw_data/all_insider_trades.parquet', index=False)  
